In [45]:
import pandas as pd
from math import sin, cos, sqrt, atan2, radians

In [46]:
edges = (
    pd.read_csv("../data/edges.csv")
    .rename(columns=lambda x: x.strip())[["# source", "target", "distance", "airline"]]
    .drop_duplicates()
)
edges = edges.set_index(["# source", "target"])
nodes = pd.read_csv("../data/nodes.csv").rename(columns=lambda x: x.strip())[
    ["name", "city", "country", "latitude", "longitude", "altitude"]
]
nodes = pd.merge(nodes, pd.read_csv("../data/continents.csv"), on="country")

In [47]:
def make_start_end(edges_df):
    new_edges = edges_df.copy().reset_index()
    new_edges[["start", "end"]] = (
        new_edges[["# source", "target"]]
        .apply(
            lambda row: f"{nodes.loc[row['# source'],'latitude']},"
            + f"{nodes.loc[row['# source'],'longitude']};"
            + f"{nodes.loc[row['target'],'latitude']},"
            + f"{nodes.loc[row['target'],'longitude']}",
            axis=1,
        )
        .str.split(";", expand=True)
    )
    return new_edges

In [49]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Radius of the Earth in km
    
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    # Haversine formula
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c

route_edges = make_start_end(edges.query("`# source` == 2374 or target == 2374"))
display(route_edges)
for i, edge in route_edges.iterrows():
    s,e = edge[["start","end"]].values
    print(calculate_distance(*map(float,s.split(",")),*map(float,e.split(",")))-edge["distance"])

,# source,target,distance,airline,start,end
0,409,2374,417.335174,P0,"-15.3308000565,28.4526004791","-12.170638697187435,26.36774594217987"
1,411,2374,265.729796,P0,"-12.998100280762,28.66489982605","-12.170638697187435,26.36774594217987"
2,2374,409,417.335174,P0,"-12.170638697187435,26.36774594217987","-15.3308000565,28.4526004791"
3,2374,411,265.729796,P0,"-12.170638697187435,26.36774594217987","-12.998100280762,28.66489982605"


-5.684341886080802e-14
0.0
-5.684341886080802e-14
0.0
